### 1. Dynamically locate project root

In [33]:
from pathlib import Path
import sys
import os
import importlib

# Locate the research directory from the notebook path or Python search path.
notebook_file = globals().get("__vsc_ipynb_file__")
search_roots = [Path.cwd(), *(Path(entry) for entry in sys.path if entry)]
if notebook_file:
    search_roots.insert(0, Path(notebook_file).resolve().parent)

PROJECT_ROOT = None
for root in search_roots:
    for current in (root.resolve(), *root.resolve().parents):
        if all((current / name).is_dir() for name in ("evaluation", "models", "datasets")):
            PROJECT_ROOT = current
            break
    if PROJECT_ROOT is not None:
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the research directory containing evaluation/, models/, and datasets/.")

REPOSITORY_ROOT = PROJECT_ROOT.parent
for import_root in (REPOSITORY_ROOT, PROJECT_ROOT):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))
# The cached engines use paths relative to the repository root.
os.chdir(PROJECT_ROOT.parent)

### Imports with the correct package path ('models.hybrid')

In [32]:
import pandas as pd
from evaluation.metrics import evaluate_model
import models.hybrid.popularity as hybrid_popularity
import models.hybrid.hybrid_engine as hybrid_engine_module
importlib.reload(hybrid_popularity)
importlib.reload(hybrid_engine_module)
HybridEngine = hybrid_engine_module.HybridEngine

meta_path = PROJECT_ROOT / "datasets" / "processed" / "meta.json"
weights_path = PROJECT_ROOT / "evaluation" / "pytorch_ncf_weights.pt"

### Load engine using explicit paths

In [30]:
engine = HybridEngine.load(weights_path=weights_path, meta_path=meta_path)
test_df = pd.read_csv(PROJECT_ROOT / "datasets" / "processed" / "test.csv")

### Run evaluation (Assign directly to `results`)

In [31]:
results = evaluate_model(
    score_fn=lambda uid, candidates: engine._ncf_scores(uid, candidates),
    test_df=test_df,
    n_items=engine.n_items,
    k=10,
    verbose=True
)

Evaluation @K=10 over 6,040 users:
  Hit@10       : 0.5290
  NDCG@10      : 0.2723
  Precision@10 : 0.0541
  Recall@10    : 0.5411
